# CassavaGuard — Verified CNN Training on Google Colab

Notebook นี้เรียก `backend/training/train_cnn.py` จากโปรเจกต์โดยตรง จึงใช้ correctness contract เดียวกับ runtime:

- TFDS `cassava:0.1.0` official train/validation/test splits
- quarantine exact-pixel duplicate groups ที่ label ขัดแย้ง
- EfficientNet-B0 ImageNet transfer learning + class weights
- checkpoint selection จาก validation macro-F1 เท่านั้น
- temperature calibration บน validation
- test evaluation หลัง model selection
- ONNX export, Keras/ONNX parity และ SHA-256 verification

> **สำคัญ:** CNN artifact จะถูกตั้ง `production_eligible=false` และต้องคง `USE_CNN=false` จนผ่าน independent Thai-field validation


## 1. เตรียมโปรเจกต์ก่อนเปิด Notebook

เลือกวิธีใดวิธีหนึ่งใน cell ตั้งค่า:

1. `upload_zip` — ZIP โปรเจกต์แล้วอัปโหลดเข้า Colab
2. `drive` — เก็บโฟลเดอร์โปรเจกต์ไว้ใน Google Drive
3. `git` — clone จาก Git repository

คำสั่งสร้าง ZIP แบบไม่รวมไฟล์ใหญ่ในเครื่อง local:

```bash
cd "/Users/norapol/Documents/vita gaurd"
zip -r cassavaguard-colab.zip cassavaguard \
  -x 'cassavaguard/backend/training/.venv/*' \
     'cassavaguard/.venv-training/*' \
     'cassavaguard/node_modules/*' \
     'cassavaguard/.git/*' \
     'cassavaguard/database/*' \
     'cassavaguard/uploads/*'
```


In [ ]:
#@title 2. ตั้งค่า Training
SOURCE_MODE = "upload_zip"  #@param ["upload_zip", "drive", "git"]
GIT_REPO_URL = ""  #@param {type:"string"}
GIT_BRANCH = "main"  #@param {type:"string"}
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/cassavaguard"  #@param {type:"string"}
TFDS_DATA_DIR = "/content/tensorflow_datasets"  #@param {type:"string"}
EPOCHS_HEAD = 6  #@param {type:"integer"}
EPOCHS_FINE = 10  #@param {type:"integer"}
FINE_TUNE_LAYERS = 40  #@param {type:"integer"}
BATCH_SIZE = 32  #@param {type:"integer"}
PATIENCE = 3  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
REQUIRE_GPU = True  #@param {type:"boolean"}
SAVE_ARTIFACTS_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CassavaGuard/models"  #@param {type:"string"}

assert SOURCE_MODE in {"upload_zip", "drive", "git"}
assert EPOCHS_HEAD >= 0 and EPOCHS_FINE >= 0 and EPOCHS_HEAD + EPOCHS_FINE > 0
assert min(FINE_TUNE_LAYERS, BATCH_SIZE, PATIENCE) > 0


In [ ]:
#@title 3. นำโปรเจกต์เข้า Colab
from pathlib import Path
import shutil
import subprocess
import sys

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_PATH).resolve()
elif SOURCE_MODE == "git":
    if not GIT_REPO_URL:
        raise ValueError("กรุณากำหนด GIT_REPO_URL")
    PROJECT_DIR = Path("/content/cassavaguard")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                    GIT_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("กรุณาอัปโหลด ZIP ของโปรเจกต์จำนวน 1 ไฟล์")
    extract_dir = Path("/content/cassavaguard-upload")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    shutil.unpack_archive(f"/content/{zip_names[0]}", extract_dir)
    matches = list(extract_dir.rglob("backend/training/train_cnn.py"))
    if len(matches) != 1:
        raise RuntimeError(f"หา backend/training/train_cnn.py ไม่พบหรือพบซ้ำ: {matches}")
    PROJECT_DIR = matches[0].parents[2]

required = [
    PROJECT_DIR / "requirements-training.txt",
    PROJECT_DIR / "backend/training/train_cnn.py",
    PROJECT_DIR / "backend/training/verify_artifacts.py",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"โปรเจกต์ไม่ครบ: {missing}")
print("PROJECT_DIR =", PROJECT_DIR)


In [ ]:
#@title 4. ติดตั้ง Training Dependencies
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-r", str(PROJECT_DIR / "requirements-training.txt")], check=True)
print("ติดตั้ง dependencies สำเร็จ")
print("หาก Colab แจ้งให้ Restart session ให้กด Restart แล้วรันใหม่ตั้งแต่ cell 2")


In [ ]:
#@title 5. ตรวจ GPU และ Environment
import json
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

gpus = tf.config.list_physical_devices("GPU")
environment = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "tensorflow_datasets": tfds.__version__,
    "gpus": [gpu.name for gpu in gpus],
}
print(json.dumps(environment, indent=2))
if REQUIRE_GPU and not gpus:
    raise RuntimeError("ไม่พบ GPU: ไปที่ Runtime > Change runtime type > T4 GPU")


In [ ]:
#@title 6. เตรียม TFDS Cache
from pathlib import Path

if TFDS_DATA_DIR.startswith("/content/drive/"):
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
Path(TFDS_DATA_DIR).mkdir(parents=True, exist_ok=True)
print("TFDS cache =", TFDS_DATA_DIR)


In [ ]:
#@title 7. Train EfficientNet-B0 CNN
import subprocess
import sys
import time

command = [
    sys.executable, str(PROJECT_DIR / "backend/training/train_cnn.py"),
    "--epochs-head", str(EPOCHS_HEAD),
    "--epochs-fine", str(EPOCHS_FINE),
    "--fine-tune-layers", str(FINE_TUNE_LAYERS),
    "--batch-size", str(BATCH_SIZE),
    "--patience", str(PATIENCE),
    "--seed", str(SEED),
    "--data-dir", TFDS_DATA_DIR,
]
print("Running:", " ".join(command))
started = time.time()
subprocess.run(command, cwd=PROJECT_DIR, check=True)
print(f"Training completed in {(time.time() - started) / 60:.1f} minutes")


In [ ]:
#@title 8. Verify CNN Artifact
import subprocess
import sys

verify_command = [
    sys.executable, str(PROJECT_DIR / "backend/training/verify_artifacts.py"),
    "--cnn-only", "--require-cnn",
]
subprocess.run(verify_command, cwd=PROJECT_DIR, check=True)


In [ ]:
#@title 9. แสดงผล Validation/Test
import json
from pathlib import Path

MODEL_DIR = PROJECT_DIR / "backend/ml_models"
metrics = json.loads((MODEL_DIR / "cnn_metrics.json").read_text())
summary = {
    "model_id": metrics["model_id"],
    "production_eligible": metrics["production_eligible"],
    "effective_split_counts": metrics["dataset"]["effective_split_counts"],
    "validation": metrics["validation"],
    "test": metrics["test"],
    "temperature": metrics["temperature"],
    "onnx_parity": metrics["onnx_parity"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
#@title 10. บันทึกและดาวน์โหลด Artifacts
from pathlib import Path
import shutil
import zipfile

artifact_names = [
    "cnn_efficientnet_b0.keras",
    "cnn_primary.onnx",
    "cnn_metrics.json",
]
for name in artifact_names:
    path = MODEL_DIR / name
    if not path.is_file():
        raise FileNotFoundError(path)

archive = Path("/content/cassavaguard_cnn_artifacts.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in artifact_names:
        bundle.write(MODEL_DIR / name, arcname=name)

if SAVE_ARTIFACTS_TO_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    drive_output = Path(DRIVE_OUTPUT_DIR)
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_output / archive.name)
    for name in artifact_names:
        shutil.copy2(MODEL_DIR / name, drive_output / name)
    print("Saved to", drive_output)

from google.colab import files
files.download(str(archive))


## 11. นำโมเดลกลับเข้าโปรเจกต์

แตก ZIP แล้ววางไฟล์ทั้งสามใน `backend/ml_models/`:

- `cnn_efficientnet_b0.keras` — framework checkpoint
- `cnn_primary.onnx` — runtime artifact
- `cnn_metrics.json` — preprocessing/calibration/hash contract

จากนั้นตรวจในเครื่อง local:

```bash
.venv-training/bin/python backend/training/verify_artifacts.py --require-cnn
```

อย่าเปิด `USE_CNN=true` ใน production จากคะแนน TFDS เพียงชุดเดียว ต้องผ่าน Thai-field holdout, per-class recall, calibration และ OOD gate ตาม `docs/TRAINING.md` ก่อน
